In [2]:
import sys
import os
os.chdir("/home/felixpersson/MasterThesisProject")
print(os.getcwd())

/home/felixpersson/MasterThesisProject


In [3]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import json

from models.neutronics.axial_neutron_model import NeutronicsModel
import data.dataclass as d_class

%config InlineBackend.figure_format = "retina"

Mesh warning: N_Z changed from 50 to 46
Mesh warning: Neutronics N_Z changed from 50 to 30
Mesh warning: N_R changed from 15 to 65


In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

from utils.solver import Solver

mesh = d_class.NeutronicsMesh(
    N_R = 50,
    N_Z = 100,
    l = 1.8
)
energy = d_class.NeutronicsEnergy(
    N_G = 8,
    power = 1000
)
cfg = d_class.NeutronicsConfig(mesh, energy)

MODEL_PATH = Path("./utils/rgi_surrogate.joblib")
interpolator_model = OpenMCTallyGridSurrogate()
interpolator_model = interpolator_model.load(MODEL_PATH)

neutronics_model = NeutronicsModel(cfg)
neutronics_model.interpolator_model = interpolator_model

solver = Solver([neutronics_model])
solver.fsolve()

phi_n_g, k = solver.solution      # type: ignore

mpl.rcParams["text.usetex"] = True
mpl.rcParams["font.family"] = "Computer modern"
mpl.rcParams["font.size"] = 22

fig = plt.figure(figsize=(16,9))
ax = fig.add_subplot(111)

colors = plt.cm.viridis(np.linspace(0, 1, 8))[::-1] # type: ignore
ax.grid(alpha=0.4)
for i in range(len(colors)):
    ax.plot(phi_n_g[:, i], color=colors[i], label=i)
ax.legend()
ax.set_yscale("log")
ax.set_title(r"$k_{eff} = $" + f"{k:.2f}")

plt.show()